### Imports + Load Data

In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Load NFHS dataset
df = pd.read_csv("../data/processed/state_factors_nfhs.csv")

df.head()

,state,immunization,stunting,wasting,female_literacy,sanitation
0,Andaman & Nicobar Islands,77.78,22.47,15.99,83.52,87.97
1,Andhra Pradesh,73.02,31.16,16.06,65.57,77.26
2,Arunachal Pradesh,64.87,27.98,13.08,71.16,82.88
3,Assam,66.44,35.29,21.73,78.19,68.55
4,Bihar,70.96,42.94,22.89,61.10,49.40


In [4]:
features = ['immunization', 'stunting', 'wasting', 'female_literacy', 'sanitation']

df[features].head()

,immunization,stunting,wasting,female_literacy,sanitation
0,77.78,22.47,15.99,83.52,87.97
1,73.02,31.16,16.06,65.57,77.26
2,64.87,27.98,13.08,71.16,82.88
3,66.44,35.29,21.73,78.19,68.55
4,70.96,42.94,22.89,61.10,49.40


In [5]:
scaler = MinMaxScaler()

df_scaled = df.copy()
df_scaled[features] = scaler.fit_transform(df[features])

df_scaled.head()

,state,immunization,stunting,wasting,female_literacy,sanitation
0,Andaman & Nicobar Islands,0.905535,0.093750,0.440954,0.653356,0.794189
1,Andhra Pradesh,0.879240,0.420934,0.445026,0.133970,0.607864
2,Arunachal Pradesh,0.834217,0.301205,0.271670,0.295718,0.705637
3,Assam,0.842890,0.576431,0.774869,0.499132,0.456333
4,Bihar,0.867860,0.864458,0.842350,0.004630,0.123173


### Risk Score (Basic + Weighted)

In [6]:
df_scaled['risk_score'] = (
    df_scaled['stunting'] +
    df_scaled['wasting'] +
    (1 - df_scaled['immunization']) +
    (1 - df_scaled['female_literacy']) +
    (1 - df_scaled['sanitation'])
)

df_scaled[['state', 'risk_score']].head()

,state,risk_score
0,Andaman & Nicobar Islands,1.181623
1,Andhra Pradesh,2.244887
2,Arunachal Pradesh,1.737303
3,Assam,2.552945
4,Bihar,3.711145


In [10]:
df_scaled['weighted_risk_score'] = (
    0.3 * df_scaled['stunting'] +
    0.3 * df_scaled['wasting'] +
    0.15 * (1 - df_scaled['immunization']) +
    0.15 * (1 - df_scaled['female_literacy']) +
    0.1 * (1 - df_scaled['sanitation'])
)

df_scaled[['state', 'weighted_risk_score']].head()

,state,weighted_risk_score
0,Andaman & Nicobar Islands,0.247159
1,Andhra Pradesh,0.447020
2,Arunachal Pradesh,0.331808
3,Assam,0.558453
4,Bihar,0.768852


In [11]:
df_scaled['risk_category'] = pd.qcut(
    df_scaled['weighted_risk_score'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

df_scaled[['state', 'weighted_risk_score', 'risk_category']].head()

,state,weighted_risk_score,risk_category
0,Andaman & Nicobar Islands,0.247159,Low
1,Andhra Pradesh,0.447020,Medium
2,Arunachal Pradesh,0.331808,Medium
3,Assam,0.558453,High
4,Bihar,0.768852,High


In [12]:
df_scaled['risk_category'].value_counts()

risk_category
Low       12
Medium    12
High      12
Name: count, dtype: int64

In [13]:
df_scaled.to_csv("../data/processed/nfhs_with_risk_scores.csv", index=False)